# 01 Synthetic Dataset Generation

This notebook creates the RGB synthetic naartjie defect dataset used by the later colour-space and YOLO notebooks.

Run this first. It reads transparent foreground defect PNGs and naartjie background images, composites the defects onto fruit regions, and writes YOLO segmentation labels plus COCO annotations.


## Setup

The shared project configuration lives in `config/project.yaml`. Dataset size, class names, image size, split ratios, and synthetic augmentation settings are controlled there.


In [ ]:
%load_ext autoreload
%autoreload 2

from pathlib import Path
import json

from naartjie_cv.config import class_names, load_config, project_root, resolve_path

ROOT = project_root()
CONFIG = ROOT / 'config' / 'project.yaml'
cfg = load_config(CONFIG)

print(f'Project root: {ROOT}')
print(f'Config: {CONFIG}')


## Input Check

Expected input folders:

- `data/raw/foreground/class_1/*.png`
- `data/raw/foreground/class_2/*.png`
- `data/raw/foreground/class_3/*.png`
- `data/raw/background/*`

Foreground PNG alpha channels become the instance masks.


In [ ]:
foreground_dir = resolve_path(cfg, cfg['paths']['foreground_dir'])
background_dir = resolve_path(cfg, cfg['paths']['background_dir'])
image_exts = {'.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff', '.webp'}

print(f'Foreground root: {foreground_dir}')
for class_id, name in class_names(cfg).items():
    count = len(list((foreground_dir / name).glob('*.png')))
    print(f'  {class_id}: {name}: {count} transparent PNG files')

background_count = sum(1 for p in background_dir.rglob('*') if p.is_file() and p.suffix.lower() in image_exts)
print(f'Background images: {background_count}')


## Generate Synthetic RGB Dataset

Outputs are written under `data/synthetic/rgb`:

- `images/train`, `images/val`, `images/test`
- `labels/train`, `labels/val`, `labels/test`
- `coco/instances_train.json`, `instances_val.json`, `instances_test.json`
- `data.yaml`


In [ ]:
from naartjie_cv.synthetic import generate_synthetic_dataset

rgb_summary = generate_synthetic_dataset(CONFIG)
print(json.dumps(rgb_summary, indent=2))


## Next Notebook

After this finishes, continue with `02_colour_space_rgb_hsv_experiment.ipynb` to create the RGB+HSV dataset and run colour-space experiments, or go directly to `03_yolo_training_evaluation.ipynb` for the main RGB YOLO training run.
